# Upwork — Early Signal Hiring Predictor
**Features:** Full client history · Keywords (TF-IDF) · Experience level · Activity (Interviewing / Invites sent / Unanswered invites)  
**Excluded:** Proposals, last-viewed timestamp, budget signals  
**Target:** `hired_bin` — did this job result in at least one hire? (0/1)  
**Models:** Logistic Regression · Random Forest · Gradient Boosting · XGBoost · LightGBM

In [1]:
import subprocess, sys

packages = {'xgboost': 'xgboost', 'lightgbm': 'lightgbm'}
for pip_name, import_name in packages.items():
    try:
        __import__(import_name)
    except ImportError:
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', pip_name, '-q'])

print('All packages ready')

All packages ready


In [2]:
import os
if os.environ.get('MPLBACKEND'):
    del os.environ['MPLBACKEND']

import matplotlib
matplotlib.use('Agg')

import pandas as pd
import numpy as np
import warnings
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics import (classification_report, confusion_matrix,
                              roc_auc_score, f1_score, precision_score,
                              recall_score, accuracy_score, precision_recall_curve)
import xgboost as xgb
import lightgbm as lgb

warnings.filterwarnings('ignore')
pd.set_option('display.float_format', '{:.4f}'.format)

DATA_PATH = Path(r'C:\Relu\upwork_bot\Model\Dataset\notification_data.xlsx')
print(f'Imports OK | Dataset: {DATA_PATH.exists()}')

Imports OK | Dataset: True


## 1. Load Data

In [3]:
df = pd.read_excel(DATA_PATH)
print(f'Shape: {df.shape}')
print(f'Columns: {df.columns.tolist()}')
df.head(3)

Shape: (1777, 33)
Columns: ['scrap_date', 'job_link', 'job_id', 'job_title', 'job_description', 'job_type', 'experience_level', 'estimated_budget', 'duration', 'keywords', 'activity_data', 'client_data', 'filter_by', 'filter_reason', 'status', 'applied_by', 'proposal_min', 'proposal_max', 'last_viewed_client', 'interviewing', 'invites_sent', 'unanswered_invites', 'hired', 'scrap_date_parsed', 'scrap_day', 'client_location', 'client_total_spent', 'client_hire_rate', 'client_hires', 'active', 'hourly_rate', 'client_jobs_posted', 'open_jobs']


,scrap_date,job_link,job_id,job_title,job_description,job_type,experience_level,estimated_budget,duration,keywords,...,scrap_date_parsed,scrap_day,client_location,client_total_spent,client_hire_rate,client_hires,active,hourly_rate,client_jobs_posted,open_jobs
0,16-05-2026 15:05,https://www.upwork.com/jobs/~02205429537621724...,022054295376217244110,LinkedIn Lead Generation Support - Lead Genera...,I am looking for a reliable freelancer to assi...,Fixed-price,Intermediate,60,NaN,"Lead Generation, LinkedIn Recruiting, LinkedIn...",...,2026-05-16 15:05:00,16-05,Serbia,521.0000,73,13,1,14.5000,18,0
1,16-05-2026 15:05,https://www.upwork.com/jobs/~02205429144905979...,022054291449059790551,Email List Building Specialist - Lead Generati...,We are seeking a skilled freelancer to build a...,Fixed-price,Intermediate,10,NaN,"Email Marketing, Email Communication, Data Ent...",...,2026-05-16 15:05:00,16-05,United States,3800.0000,100,190,23,4.1400,182,25
2,16-05-2026 15:06,https://www.upwork.com/jobs/~02205424489753791...,022054244897537917390,"I need a list of childcare professionals, teac...",Here is the updated job description:\n\nPlease...,Fixed-price,Intermediate,50,NaN,"English, Data Entry, Communications, Lead Gene...",...,2026-05-16 15:06:00,16-05,United States,82000.0000,63,373,5,6.1200,497,6


## 2. Define Target
`hired` is the count of people hired. Binarised: **1 = at least one hire, 0 = no hire**.

In [4]:
df['hired_bin'] = (df['hired'] > 0).astype(int)

print('Class distribution:')
vc = df['hired_bin'].value_counts()
print(vc)
print(f'\nHire rate             : {df["hired_bin"].mean():.1%}')
print(f'Imbalance ratio neg:pos: {vc[0]/vc[1]:.1f}:1')

Class distribution:
hired_bin
1    976
0    801
Name: count, dtype: int64

Hire rate             : 54.9%
Imbalance ratio neg:pos: 0.8:1


## 3. Drop Irrelevant Columns
Identifiers, leaky columns, raw JSON, proposal counts, and last-viewed timestamp are all removed.
This model deliberately excludes proposals and last-viewed — predicting from client history + activity only.

In [5]:
drop_cols = [
    'job_id', 'job_link',                          # identifiers
    'scrap_date', 'scrap_date_parsed', 'scrap_day', # dates / zero-variance
    'applied_by', 'filter_by',                      # 100% missing
    'filter_reason', 'status',                      # 100% missing
    'hired',                                        # replaced by hired_bin
    'activity_data', 'client_data',                 # raw JSON (already parsed)
    'job_type', 'duration',                         # low signal
    'job_title', 'job_description',                 # excluded — text not in scope
    # Excluded from this model:
    'proposal_min', 'proposal_max',                 # proposal counts — not used
    'last_viewed_client',                           # last-viewed — not used
    'estimated_budget',                             # budget — not used
]
df = df.drop(columns=[c for c in drop_cols if c in df.columns])
print(f'Shape after drop: {df.shape}')
print('Remaining columns:', df.columns.tolist())

Shape after drop: (1777, 14)
Remaining columns: ['experience_level', 'keywords', 'interviewing', 'invites_sent', 'unanswered_invites', 'client_location', 'client_total_spent', 'client_hire_rate', 'client_hires', 'active', 'hourly_rate', 'client_jobs_posted', 'open_jobs', 'hired_bin']


## 4. Handle Missing Values

In [6]:
num_cols = df.select_dtypes(include='number').columns.tolist()

for col in num_cols:
    df[col] = df[col].fillna(df[col].median())

df['client_location'] = df['client_location'].fillna('unknown')
df['keywords']        = df['keywords'].fillna('').astype(str)
df['experience_level'] = df['experience_level'].fillna('intermediate')

print('Missing after fill:')
m = df.isnull().sum()
print(m[m > 0].to_string() if m.sum() > 0 else 'None')

Missing after fill:
None


In [7]:
# Split BEFORE any encoding so encoders are fitted on train rows only
train_idx, test_idx = train_test_split(
    df.index, test_size=0.2, random_state=42, stratify=df['hired_bin']
)
print(f'Train: {len(train_idx)} rows | hired={df.loc[train_idx, "hired_bin"].sum()}')
print(f'Test : {len(test_idx)}  rows | hired={df.loc[test_idx,  "hired_bin"].sum()}')

Train: 1421 rows | hired=780
Test : 356  rows | hired=196


## 5. Feature Engineering

**Feature groups in this model:**
1. **Activity** — `interviewing`, `invites_sent`, `unanswered_invites` + binary/ratio derivatives
2. **Client history** — spend, hire rate, hires, capacity, open jobs
3. **Experience level** — ordinal + keyword/level alignment mismatch
4. **Keywords** — TF-IDF (fitted in section 7)

In [8]:
# ── Activity features (3 raw + 3 derived) ─────────────────────────────────────
df['has_interviews']   = (df['interviewing'] > 0).astype(int)
df['has_invites']      = (df['invites_sent'] > 0).astype(int)
answered_inv           = (df['invites_sent'] - df['unanswered_invites']).clip(lower=0)
df['invite_reply_rate'] = answered_inv / (df['invites_sent'] + 1)

# ── Client history features ────────────────────────────────────────────────────
df['client_is_reliable'] = (df['client_hire_rate'] >= 80).astype(int)
df['log_client_hires']   = np.log1p(df['client_hires'])
df['log_client_spent']   = np.log1p(df['client_total_spent'])
df['client_is_new']      = ((df['client_hires'] == 0) & (df['client_total_spent'] == 0)).astype(int)
df['hiring_capacity']    = 1 / (df['active'] + 1)
df['single_open_job']    = (df['open_jobs'] == 1).astype(int)
df['many_open_jobs']     = (df['open_jobs'] >= 5).astype(int)
df['open_jobs_capped']   = df['open_jobs'].clip(upper=5)

# ── Experience level ordinal ───────────────────────────────────────────────────
_exp_map = {'entry level': 1, 'entry': 1, 'intermediate': 2, 'expert': 3}
df['exp_level_ord'] = (
    df['experience_level'].str.lower().str.strip().map(_exp_map).fillna(2)
)

# ── Keyword–level alignment (fitted on train rows only) ───────────────────────
_train_kw = df.loc[train_idx, ['keywords', 'exp_level_ord']].copy()
_train_kw['kw_list'] = _train_kw['keywords'].str.split(',').apply(
    lambda x: [k.strip().lower() for k in x if k.strip()])
kw_level_map = (
    _train_kw.explode('kw_list')
             .query("kw_list != ''")
             .groupby('kw_list')['exp_level_ord']
             .mean()
)

def _keyword_avg_level(kw_str, default=2.0):
    if not kw_str or str(kw_str).strip() == '':
        return default
    kws = [k.strip().lower() for k in str(kw_str).split(',') if k.strip()]
    levels = [kw_level_map[k] for k in kws if k in kw_level_map]
    return float(np.mean(levels)) if levels else default

df['keyword_avg_level'] = df['keywords'].apply(_keyword_avg_level)
df['level_mismatch']    = np.abs(df['exp_level_ord'] - df['keyword_avg_level'])

print('Feature engineering done.')
print(f'  interviewing > 0  : {df["has_interviews"].sum()} ({df["has_interviews"].mean():.1%})')
print(f'  invites_sent > 0  : {df["has_invites"].sum()} ({df["has_invites"].mean():.1%})')
print(f'  exp_level_ord     : {df["exp_level_ord"].value_counts().to_dict()}')
print(f'  level_mismatch    : mean={df["level_mismatch"].mean():.2f} max={df["level_mismatch"].max():.2f}')

Feature engineering done.
  interviewing > 0  : 937 (52.7%)
  invites_sent > 0  : 800 (45.0%)
  exp_level_ord     : {2: 1060, 3: 562, 1: 155}
  level_mismatch    : mean=0.41 max=1.53


## 6. Encode Client Location
Top-10 locations dummy-encoded; fitted on train rows only.

In [9]:
loc_abbrev = {
    'usa': 'united states', 'u.s.': 'united states',
    'gbr': 'united kingdom', 'u.k.': 'united kingdom',
    'aus': 'australia', 'nld': 'netherlands',
    'can': 'canada', 'deu': 'germany', 'ind': 'india',
}
df['client_location_norm'] = (
    df['client_location'].fillna('unknown').str.lower()
      .map(lambda x: loc_abbrev.get(x, x))
)
top_locs = df.loc[train_idx, 'client_location_norm'].value_counts().head(10).index.tolist()
df['client_location_grp'] = df['client_location_norm'].apply(
    lambda x: x if x in top_locs else 'other'
)
loc_dummies = pd.get_dummies(df['client_location_grp'], prefix='loc')
df = pd.concat([df, loc_dummies], axis=1)

print('Location encoding done (fitted on train rows only).')
print('Dummies:', loc_dummies.columns.tolist())

Location encoding done (fitted on train rows only).
Dummies: ['loc_australia', 'loc_canada', 'loc_germany', 'loc_india', 'loc_netherlands', 'loc_nigeria', 'loc_other', 'loc_pakistan', 'loc_united arab emirates', 'loc_united kingdom', 'loc_united states']


## 7. TF-IDF on Keywords
30 keyword features fitted on training rows only.

In [10]:
vec_kw = TfidfVectorizer(
    max_features=30, ngram_range=(1, 2),
    stop_words='english', min_df=2, sublinear_tf=True
)
vec_kw.fit(df.loc[train_idx, 'keywords'].fillna('').astype(str))
mat_kw = vec_kw.transform(df['keywords'].fillna('').astype(str))
tfidf_kw = pd.DataFrame(mat_kw.toarray(),
                         columns=[f'kw_{c}' for c in vec_kw.get_feature_names_out()],
                         index=df.index)
df = pd.concat([df, tfidf_kw], axis=1)

print(f'Keyword TF-IDF: {tfidf_kw.shape[1]} features')

Keyword TF-IDF: 30 features


## 8. Build Feature Matrix

| Group | Features |
|---|---|
| Activity | `interviewing`, `invites_sent`, `unanswered_invites`, `has_interviews`, `has_invites`, `invite_reply_rate` |
| Client history | hire rate, hires (raw + log), spent (raw + log), new flag, jobs posted, active, capacity, open jobs |
| Experience level | `exp_level_ord`, `keyword_avg_level`, `level_mismatch` |
| Location | 11 dummy columns |
| Keywords | 30 TF-IDF features |

In [11]:
loc_cols = [c for c in df.columns if c.startswith('loc_')]
kw_cols  = [c for c in df.columns if c.startswith('kw_')]

all_features = [
    # Activity (6)
    'interviewing', 'invites_sent', 'unanswered_invites',
    'has_interviews', 'has_invites', 'invite_reply_rate',
    # Client history (13)
    'client_hire_rate', 'client_is_reliable',
    'client_hires', 'log_client_hires',
    'client_total_spent', 'log_client_spent', 'client_is_new',
    'client_jobs_posted', 'active', 'hiring_capacity',
    'single_open_job', 'many_open_jobs', 'open_jobs_capped',
    # Experience level (3)
    'exp_level_ord', 'keyword_avg_level', 'level_mismatch',
    # Location dummies (~11)
    *loc_cols,
    # Keyword TF-IDF (30)
    *kw_cols,
]

y = df['hired_bin'].copy()
X = df[all_features].fillna(0).astype(float)

dups = [c for c in all_features if all_features.count(c) > 1]
assert not dups, f'Duplicate features: {set(dups)}'
assert X.isnull().sum().sum() == 0, 'NaN in X'

print(f'Features : {X.shape[1]}')
print(f'Samples  : {len(y)} | hired={y.sum()} ({y.mean():.1%})')
print(f'\nFeature groups:')
print(f'  Activity      : 6')
print(f'  Client history: 13')
print(f'  Exp level     : 3')
print(f'  Location      : {len(loc_cols)}')
print(f'  Keywords TF-IDF: {len(kw_cols)}')

Features : 63
Samples  : 1777 | hired=976 (54.9%)

Feature groups:
  Activity      : 6
  Client history: 13
  Exp level     : 3
  Location      : 11
  Keywords TF-IDF: 30


## 9. Apply Train/Test Split

In [12]:
X_train = X.loc[train_idx]
X_test  = X.loc[test_idx]
y_train = y.loc[train_idx]
y_test  = y.loc[test_idx]

neg_count = (y_train == 0).sum()
pos_count = (y_train == 1).sum()
ratio = neg_count / pos_count

print(f'Train: {len(y_train)} | hired={pos_count} ({y_train.mean():.1%}) | neg:pos={ratio:.2f}:1')
print(f'Test : {len(y_test)}  | hired={y_test.sum()} ({y_test.mean():.1%})')

Train: 1421 | hired=780 (54.9%) | neg:pos=0.82:1
Test : 356  | hired=196 (55.1%)


## 10. Train Models

In [13]:
results = []
fitted  = {}

def evaluate(name, model, store_key=None):
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)
    y_prob = model.predict_proba(X_test)[:, 1]
    r = {
        'Model'    : name,
        'Accuracy' : accuracy_score(y_test, y_pred),
        'Precision': precision_score(y_test, y_pred, zero_division=0),
        'Recall'   : recall_score(y_test, y_pred, zero_division=0),
        'F1'       : f1_score(y_test, y_pred, zero_division=0),
        'ROC-AUC'  : roc_auc_score(y_test, y_prob),
    }
    print(f'\n--- {name} ---')
    print(classification_report(y_test, y_pred, target_names=['Not Hired', 'Hired']))
    if store_key:
        fitted[store_key] = model
    return r

neg, pos = (y_train == 0).sum(), (y_train == 1).sum()
print(f'neg:pos = {neg}:{pos}  →  scale_pos_weight = {neg/pos:.2f}')

neg:pos = 641:780  →  scale_pos_weight = 0.82


In [14]:
results.append(evaluate('Logistic Regression',
    LogisticRegression(class_weight='balanced', max_iter=1000,
                       C=1.0, solver='lbfgs', random_state=42)))


--- Logistic Regression ---
              precision    recall  f1-score   support

   Not Hired       0.68      0.67      0.68       160
       Hired       0.73      0.74      0.74       196

    accuracy                           0.71       356
   macro avg       0.71      0.71      0.71       356
weighted avg       0.71      0.71      0.71       356



In [15]:
results.append(evaluate('Random Forest',
    RandomForestClassifier(n_estimators=300, class_weight='balanced',
                           min_samples_leaf=5, random_state=42, n_jobs=-1),
    store_key='rf'))


--- Random Forest ---
              precision    recall  f1-score   support

   Not Hired       0.68      0.72      0.70       160
       Hired       0.76      0.72      0.74       196

    accuracy                           0.72       356
   macro avg       0.72      0.72      0.72       356
weighted avg       0.72      0.72      0.72       356



In [16]:
sample_w = np.where(y_train == 1, neg / pos, 1.0)
gb = GradientBoostingClassifier(n_estimators=200, learning_rate=0.05,
                                 max_depth=4, subsample=0.8, random_state=42)
gb.fit(X_train, y_train, sample_weight=sample_w)
fitted['gb'] = gb

y_pred = gb.predict(X_test)
y_prob = gb.predict_proba(X_test)[:, 1]
print('\n--- Gradient Boosting ---')
print(classification_report(y_test, y_pred, target_names=['Not Hired', 'Hired']))
results.append({'Model': 'Gradient Boosting',
                'Accuracy' : accuracy_score(y_test, y_pred),
                'Precision': precision_score(y_test, y_pred, zero_division=0),
                'Recall'   : recall_score(y_test, y_pred, zero_division=0),
                'F1'       : f1_score(y_test, y_pred, zero_division=0),
                'ROC-AUC'  : roc_auc_score(y_test, y_prob)})


--- Gradient Boosting ---
              precision    recall  f1-score   support

   Not Hired       0.66      0.61      0.64       160
       Hired       0.70      0.74      0.72       196

    accuracy                           0.69       356
   macro avg       0.68      0.68      0.68       356
weighted avg       0.68      0.69      0.68       356



In [17]:
results.append(evaluate('XGBoost',
    xgb.XGBClassifier(n_estimators=300, learning_rate=0.05, max_depth=4,
                       subsample=0.8, colsample_bytree=0.8,
                       scale_pos_weight=neg/pos, eval_metric='logloss',
                       use_label_encoder=False, random_state=42, n_jobs=-1),
    store_key='xgb'))


--- XGBoost ---
              precision    recall  f1-score   support

   Not Hired       0.66      0.62      0.64       160
       Hired       0.71      0.74      0.72       196

    accuracy                           0.69       356
   macro avg       0.68      0.68      0.68       356
weighted avg       0.69      0.69      0.69       356



In [18]:
results.append(evaluate('LightGBM',
    lgb.LGBMClassifier(n_estimators=300, learning_rate=0.05, max_depth=4,
                        subsample=0.8, colsample_bytree=0.8,
                        class_weight='balanced', random_state=42, n_jobs=-1, verbose=-1),
    store_key='lgb'))


--- LightGBM ---
              precision    recall  f1-score   support

   Not Hired       0.66      0.64      0.65       160
       Hired       0.71      0.73      0.72       196

    accuracy                           0.69       356
   macro avg       0.69      0.69      0.69       356
weighted avg       0.69      0.69      0.69       356



## 11. Model Comparison

In [19]:
df_results = pd.DataFrame(results).set_index('Model')
print('=== MODEL COMPARISON (Early Signal Model) ===')
print((df_results * 100).round(2).to_string())

fig, axes = plt.subplots(1, 2, figsize=(13, 4))
models = df_results.index.tolist()
x = np.arange(len(models))

for ax, metric in zip(axes, ['F1', 'ROC-AUC']):
    bars = ax.bar(x, df_results[metric], color='#4C72B0', alpha=0.85)
    ax.set_xticks(x); ax.set_xticklabels(models, rotation=15, ha='right')
    ax.set_ylim(0, 1); ax.set_title(metric, fontweight='bold')
    ax.bar_label(bars, fmt='%.3f', padding=2, fontsize=9)

plt.suptitle('Early Signal Model — Upwork Hiring Predictor', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig(r'C:\Relu\upwork_bot\Model\Dataset\early_model_comparison.png', dpi=120, bbox_inches='tight')
plt.show()

=== MODEL COMPARISON (Early Signal Model) ===
                     Accuracy  Precision  Recall      F1  ROC-AUC
Model                                                            
Logistic Regression   71.0700    73.3700 74.4900 73.9200  76.9200
Random Forest         72.1900    75.9400 72.4500 74.1500  79.9000
Gradient Boosting     68.5400    70.1900 74.4900 72.2800  78.3600
XGBoost               68.8200    70.7300 73.9800 72.3200  79.3900
LightGBM              69.1000    71.5000 72.9600 72.2200  78.4200


## 12. Feature Importance

In [20]:
fi_rf  = pd.Series(fitted['rf'].feature_importances_,  index=all_features)
fi_xgb = pd.Series(fitted['xgb'].feature_importances_, index=all_features)

for name, fi, color in [('Random Forest', fi_rf, '#4C72B0'), ('XGBoost', fi_xgb, '#DD8452')]:
    top = fi.sort_values(ascending=False).head(20)
    fig, ax = plt.subplots(figsize=(9, 6))
    top.sort_values().plot.barh(ax=ax, color=color, alpha=0.85)
    ax.set_title(f'{name} — Top 20 Features (Early Model)', fontweight='bold')
    ax.set_xlabel('Importance')
    plt.tight_layout()
    plt.savefig(fr'C:\Relu\upwork_bot\Model\Dataset\early_fi_{name.lower().replace(" ","_")}.png',
                dpi=120, bbox_inches='tight')
    plt.show()
    print(f'Top 10 {name}:')
    print(top.head(10).to_string())

Top 10 Random Forest:
client_hire_rate     0.1329
client_is_reliable   0.0729
client_hires         0.0549
log_client_hires     0.0512
client_total_spent   0.0399
log_client_spent     0.0390
interviewing         0.0383
keyword_avg_level    0.0370
client_jobs_posted   0.0359
level_mismatch       0.0352
Top 10 XGBoost:
client_is_reliable   0.2120
client_hire_rate     0.0430
single_open_job      0.0407
has_interviews       0.0230
invites_sent         0.0177
interviewing         0.0177
open_jobs_capped     0.0167
has_invites          0.0165
kw_data entry        0.0164
kw_microsoft         0.0163


In [21]:
top20_rf  = set(fi_rf.nlargest(20).index)
top20_xgb = set(fi_xgb.nlargest(20).index)
shared = sorted(top20_rf & top20_xgb)
print(f'Features in top-20 of BOTH RF and XGBoost ({len(shared)}):')
for f in shared:
    print(f'  {f:40s}  RF={fi_rf[f]:.4f}  XGB={fi_xgb[f]:.4f}')

Features in top-20 of BOTH RF and XGBoost (14):
  client_hire_rate                          RF=0.1329  XGB=0.0430
  client_hires                              RF=0.0549  XGB=0.0132
  client_is_reliable                        RF=0.0729  XGB=0.2120
  client_total_spent                        RF=0.0399  XGB=0.0128
  has_interviews                            RF=0.0221  XGB=0.0230
  has_invites                               RF=0.0174  XGB=0.0165
  hiring_capacity                           RF=0.0276  XGB=0.0156
  interviewing                              RF=0.0383  XGB=0.0177
  invite_reply_rate                         RF=0.0309  XGB=0.0163
  invites_sent                              RF=0.0327  XGB=0.0177
  kw_data entry                             RF=0.0166  XGB=0.0164
  kw_entry                                  RF=0.0192  XGB=0.0129
  log_client_hires                          RF=0.0512  XGB=0.0128
  open_jobs_capped                          RF=0.0286  XGB=0.0167


In [22]:
fig, axes = plt.subplots(1, 3, figsize=(14, 4))
for ax, (name, key) in zip(axes, [('Random Forest','rf'), ('XGBoost','xgb'), ('Gradient Boosting','gb')]):
    y_pred = fitted[key].predict(X_test)
    cm = confusion_matrix(y_test, y_pred)
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=ax,
                xticklabels=['Not Hired','Hired'], yticklabels=['Not Hired','Hired'])
    ax.set_title(name, fontweight='bold')
    ax.set_ylabel('Actual'); ax.set_xlabel('Predicted')
plt.tight_layout()
plt.savefig(r'C:\Relu\upwork_bot\Model\Dataset\early_confusion_matrices.png', dpi=120, bbox_inches='tight')
plt.show()

## 13. Best Model & Optimal Threshold

In [23]:
best_name = df_results['F1'].idxmax()
best_key  = (best_name.lower()
             .replace(' ', '_')
             .replace('gradient_boosting', 'gb')
             .replace('random_forest', 'rf')
             .replace('lightgbm', 'lgb')
             .replace('xgboost', 'xgb')
             .replace('logistic_regression', 'lr'))

print(f'=== BEST MODEL: {best_name} ===')
for m in ['Accuracy', 'Precision', 'Recall', 'F1', 'ROC-AUC']:
    print(f'  {m:10s}: {df_results.loc[best_name, m]:.4f}')

y_prob_best = fitted[best_key].predict_proba(X_test)[:, 1]
prec, rec, thresh = precision_recall_curve(y_test, y_prob_best)
f1_curve = 2 * prec[:-1] * rec[:-1] / (prec[:-1] + rec[:-1] + 1e-8)
optimal_threshold = float(thresh[f1_curve.argmax()])

print(f'\nOptimal threshold (F1-max): {optimal_threshold:.3f}')
print(f'\nAll models ranked by F1:')
print((df_results[['F1', 'ROC-AUC', 'Precision', 'Recall']] * 100).round(2).sort_values('F1', ascending=False).to_string())

=== BEST MODEL: Random Forest ===
  Accuracy  : 0.7219
  Precision : 0.7594
  Recall    : 0.7245
  F1        : 0.7415
  ROC-AUC   : 0.7990

Optimal threshold (F1-max): 0.459

All models ranked by F1:
                         F1  ROC-AUC  Precision  Recall
Model                                                  
Random Forest       74.1500  79.9000    75.9400 72.4500
Logistic Regression 73.9200  76.9200    73.3700 74.4900
XGBoost             72.3200  79.3900    70.7300 73.9800
Gradient Boosting   72.2800  78.3600    70.1900 74.4900
LightGBM            72.2200  78.4200    71.5000 72.9600


## 14. Save Pipeline

In [24]:
import joblib

PIPELINE_PATH = Path(r'C:\Relu\upwork_bot\Model\Dataset\early_pipeline.pkl')

pipeline = {
    'model'         : fitted[best_key],
    'model_name'    : best_name,
    'vec_kw'        : vec_kw,
    'all_features'  : all_features,
    'top_locs'      : top_locs,
    'kw_level_map'  : kw_level_map,
    'threshold'     : optimal_threshold,
}

joblib.dump(pipeline, PIPELINE_PATH)
print(f'Pipeline saved → {PIPELINE_PATH}')
print(f'  model    : {best_name}')
print(f'  features : {len(all_features)}')
print(f'  threshold: {optimal_threshold:.3f}')

Pipeline saved → C:\Relu\upwork_bot\Model\Dataset\early_pipeline.pkl
  model    : Random Forest
  features : 63
  threshold: 0.459


## 15. Predict New Job
Loads `early_pipeline.pkl` and scores any raw job dict using client history + keywords + exp level + activity (interviewing / invites / unanswered invites only).

In [25]:
import joblib, numpy as np, pandas as pd
from pathlib import Path

pl = joblib.load(Path(r'C:\Relu\upwork_bot\Model\Dataset\early_pipeline.pkl'))

_loc_abbrev = {
    'usa': 'united states', 'u.s.': 'united states',
    'gbr': 'united kingdom', 'u.k.': 'united kingdom',
    'aus': 'australia', 'nld': 'netherlands',
    'can': 'canada', 'deu': 'germany', 'ind': 'india',
}
_exp_map      = {'entry level': 1, 'entry': 1, 'intermediate': 2, 'expert': 3}
_loc_cols     = [f for f in pl['all_features'] if f.startswith('loc_')]
_threshold    = pl.get('threshold', 0.5)
_kw_level_map = pl.get('kw_level_map', {})

def predict_job_early(job: dict) -> dict:
    g = lambda k: float(job.get(k) or 0)

    # Activity
    interviewing       = g('interviewing')
    invites_sent       = g('invites_sent')
    unanswered_invites = g('unanswered_invites')
    has_interviews     = int(interviewing > 0)
    has_invites        = int(invites_sent > 0)
    answered_inv       = max(invites_sent - unanswered_invites, 0)
    invite_reply_rate  = answered_inv / (invites_sent + 1)

    # Client history
    client_hire_rate   = g('client_hire_rate')
    client_hires       = g('client_hires')
    client_total_spent = g('client_total_spent')
    client_jobs_posted = g('client_jobs_posted')
    active             = g('active')
    open_jobs          = g('open_jobs')
    client_is_reliable = int(client_hire_rate >= 80)
    log_client_hires   = np.log1p(client_hires)
    log_client_spent   = np.log1p(client_total_spent)
    client_is_new      = int(client_hires == 0 and client_total_spent == 0)
    hiring_capacity    = 1 / (active + 1)
    single_open_job    = int(open_jobs == 1)
    many_open_jobs     = int(open_jobs >= 5)
    open_jobs_capped   = min(open_jobs, 5)

    # Experience level
    exp_level_ord = _exp_map.get(
        str(job.get('experience_level') or 'intermediate').lower().strip(), 2)
    kw_str  = str(job.get('keywords') or '')
    kws     = [k.strip().lower() for k in kw_str.split(',') if k.strip()]
    kw_lvls = [_kw_level_map[k] for k in kws if k in _kw_level_map]
    keyword_avg_level = float(np.mean(kw_lvls)) if kw_lvls else 2.0
    level_mismatch    = abs(exp_level_ord - keyword_avg_level)

    # Location
    loc_raw  = str(job.get('client_location') or 'unknown').lower()
    loc_norm = _loc_abbrev.get(loc_raw, loc_raw)
    loc_grp  = loc_norm if loc_norm in pl['top_locs'] else 'other'
    loc_row  = {col: int(col == f'loc_{loc_grp}') for col in _loc_cols}

    # Keywords TF-IDF
    mat = pl['vec_kw'].transform([kw_str])
    kw_row = {f'kw_{c}': v for c, v in zip(pl['vec_kw'].get_feature_names_out(), mat.toarray()[0])}

    feat = {
        'interviewing': interviewing, 'invites_sent': invites_sent,
        'unanswered_invites': unanswered_invites,
        'has_interviews': has_interviews, 'has_invites': has_invites,
        'invite_reply_rate': invite_reply_rate,
        'client_hire_rate': client_hire_rate, 'client_is_reliable': client_is_reliable,
        'client_hires': client_hires, 'log_client_hires': log_client_hires,
        'client_total_spent': client_total_spent, 'log_client_spent': log_client_spent,
        'client_is_new': client_is_new, 'client_jobs_posted': client_jobs_posted,
        'active': active, 'hiring_capacity': hiring_capacity,
        'single_open_job': single_open_job, 'many_open_jobs': many_open_jobs,
        'open_jobs_capped': open_jobs_capped,
        'exp_level_ord': exp_level_ord,
        'keyword_avg_level': keyword_avg_level,
        'level_mismatch': level_mismatch,
        **loc_row, **kw_row,
    }

    X_row = pd.DataFrame([feat])[pl['all_features']].fillna(0)
    prob  = pl['model'].predict_proba(X_row)[0][1]
    pred  = int(prob >= _threshold)

    signal   = ('Likely Hired'  if prob >= 0.70 else
                'Possible Hire' if prob >= 0.55 else
                'Borderline'    if prob >= _threshold else
                'Unlikely Hired')
    decision = ('Apply' if prob >= 0.55 else
                'Watch' if prob >= _threshold else
                'Skip')

    return {
        'decision'   : decision,
        'signal'     : signal,
        'probability': round(prob, 4),
        'prediction' : pred,
    }

print(f'predict_job_early() ready — model: {pl["model_name"]} | threshold: {_threshold:.3f}')

predict_job_early() ready — model: Random Forest | threshold: 0.459


In [ ]:
sample_job = {
    'keywords'          : 'Lead Generation, LinkedIn Recruiting, LinkedIn',
    'experience_level'  : 'Intermediate',
    'interviewing'      : 49,
    'invites_sent'      : 0,
    'unanswered_invites': 0,
    'client_location'   : 'United States',
    'client_total_spent': 180,
    'client_hire_rate'  : 34,
    'client_hires'      : 3,
    'active'            : 2,
    'client_jobs_posted': 9,
    'open_jobs'         : 5,
}

result = predict_job_early(sample_job)
print("=" * 45)
print("  EARLY MODEL — JOB PREDICTION")
print("=" * 45)
print(f"  Keywords    : {sample_job['keywords'][:35]}")
print(f"  Interviewing: {sample_job['interviewing']}  |  Invites: {sample_job['invites_sent']}  |  Unanswered: {sample_job['unanswered_invites']}")
print(f"  Decision    : {result['decision']}")
print(f"  Signal      : {result['signal']}")
print(f"  Model Prob  : {result['probability']:.1%}")
print("=" * 45)

  EARLY MODEL — JOB PREDICTION
  Keywords    : Lead Generation, LinkedIn Recruitin
  Interviewing: 49  |  Invites: 0  |  Unanswered: 0
  Decision    : Skip
  Signal      : Unlikely Hired
  Model Prob  : 30.1%


In [29]:
sample_job = {
    'keywords'          : 'Data Scraping, Prospect List, List Building',
    'experience_level'  : 'Intermediate',
    'interviewing'      : 0,
    'invites_sent'      : 0,
    'unanswered_invites': 0,
    'client_location'   : 'United States America',
    'client_total_spent': 2100,
    'client_hire_rate'  : 33,
    'client_hires'      : 11,
    'active'            : 1,
    'client_jobs_posted': 34,
    'open_jobs'         : 2,
}

result = predict_job_early(sample_job)
print("=" * 45)
print("  EARLY MODEL — JOB PREDICTION")
print("=" * 45)
print(f"  Keywords    : {sample_job['keywords'][:35]}")
print(f"  Interviewing: {sample_job['interviewing']}  |  Invites: {sample_job['invites_sent']}  |  Unanswered: {sample_job['unanswered_invites']}")
print(f"  Decision    : {result['decision']}")
print(f"  Signal      : {result['signal']}")
print(f"  Model Prob  : {result['probability']:.1%}")
print("=" * 45)

  EARLY MODEL — JOB PREDICTION
  Keywords    : Data Scraping, Prospect List, List 
  Interviewing: 0  |  Invites: 0  |  Unanswered: 0
  Decision    : Skip
  Signal      : Unlikely Hired
  Model Prob  : 34.5%


In [30]:
sample_job = {
    'keywords'          : 'Data Entry, Data Scraping, List Building ,Data Mining, Prospect List, Microsoft Excel, Lead Generation',
    'experience_level'  : 'Intermediate',
    'interviewing'      : 4,
    'invites_sent'      : 17,
    'unanswered_invites': 6,
    'client_location'   : 'Australia',
    'client_total_spent': 3600,
    'client_hire_rate'  : 80,
    'client_hires'      : 54,
    'active'            : 20,
    'client_jobs_posted': 64,
    'open_jobs'         : 3,
}

result = predict_job_early(sample_job)
print("=" * 45)
print("  EARLY MODEL — JOB PREDICTION")
print("=" * 45)
print(f"  Keywords    : {sample_job['keywords'][:35]}")
print(f"  Interviewing: {sample_job['interviewing']}  |  Invites: {sample_job['invites_sent']}  |  Unanswered: {sample_job['unanswered_invites']}")
print(f"  Decision    : {result['decision']}")
print(f"  Signal      : {result['signal']}")
print(f"  Model Prob  : {result['probability']:.1%}")
print("=" * 45)

  EARLY MODEL — JOB PREDICTION
  Keywords    : Data Entry, Data Scraping, List Bui
  Interviewing: 4  |  Invites: 17  |  Unanswered: 6
  Decision    : Apply
  Signal      : Likely Hired
  Model Prob  : 80.3%
